# Search a catalog with managed PostgreSQL

Alice runs STAC FastAPI PgSTAC in `ws-alice/vc0`. She registers three NAIP Items from `docs/catalog.v1.parquet` and searches them with `pystac-client`.

The API uses Alice's managed PostgreSQL database. The `managed-postgres` Secret supplies its connection in namespace `stac-fastapi-pgstac`.

```mermaid
flowchart LR
  A[Sample GeoParquet] --> B[Notebook and pystac-client]
  B -->|register and search| P[Local Kubernetes proxy]
  D[Browser] --> P
  P --> C[STAC API in vc0]
  C --> E[Managed PostgreSQL]
```

## 1. Connect to Alice's workspace

Use an existing workspace with `vc0`, a PostgreSQL database named `catalog`, and the `managed-postgres` Secret in `stac-fastapi-pgstac`. An operator sets these up as shown in [Alice's Datalab](https://github.com/EOEPCA/workspace/blob/main/setup/examples.yaml).

From the repository root:

```bash
uv sync
uv run jupyter lab docs/getting-started/stac-fastapi-pgstac.ipynb
```

Sign in as Alice below. The code requests her `demo` profile from the Workspace API and saves the kubeconfig in a private temporary file.

If the API does not return profiles, ask the workspace operator for the `ws-alice/vc0/demo` kubeconfig. Save it outside the repository and enter its relative path when prompted. This profile needs edit access in `stac-fastapi-pgstac`. No host-cluster kubeconfig is used.

In [1]:
import json
import secrets
import subprocess
import time
from html import escape
from pathlib import Path
from urllib.parse import quote

import pyarrow.parquet as pq
import pystac
import requests
from IPython.display import HTML, display
from pystac_client import Client
from stac_geoparquet.arrow import stac_table_to_items

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "docs/catalog.v1.parquet").is_file()
)
MANIFEST_DIR = REPO_ROOT / "docs/getting-started/stac-fastapi-pgstac"
NAMESPACE = "stac-fastapi-pgstac"
LOCAL_PORT = 8002
PROXY_PATH = f"/api/v1/namespaces/{NAMESPACE}/services/catalog:8080/proxy"
ROOT_PATH = PROXY_PATH

import sys
sys.path.insert(0, str(REPO_ROOT / "docs"))
from notebook_support import load_config, workspace_login, workspace_kubeconfig

config = load_config("eoepca-demo.env", "workspace")
owner_tokens = workspace_login(config, "alice", globals().get("owner_tokens"))
profile_directory, VCLUSTER_KUBECONFIG = workspace_kubeconfig(
    config, "ws-alice", owner_tokens, instance="vc0", group="demo",
)
KUBECTL = ["kubectl", "--kubeconfig", str(VCLUSTER_KUBECONFIG)]


def kube(*args, document=None, check=True):
    result = subprocess.run(
        [*KUBECTL, "--namespace", NAMESPACE, *args],
        input=json.dumps(document) if document is not None else None,
        text=True, capture_output=True,
    )
    if check and result.returncode:
        # A command can contain Secret data. Keep raw errors out of saved output.
        raise RuntimeError(f"kubectl {args[0]} failed (exit {result.returncode}).")
    return result


def apply(document):
    return kube("apply", "-f", "-", document=document)


context = subprocess.run(
    [*KUBECTL, "config", "current-context"],
    text=True, capture_output=True, check=True,
).stdout.strip()
print({"context": context, "namespace": NAMESPACE})
if not context.startswith("ws-alice-vcluster-vc0-"):
    raise RuntimeError("Select Alice's vc0 profile, not the host cluster.")
if kube("auth", "can-i", "create", "deployments").stdout.strip() != "yes":
    raise PermissionError("The profile needs edit access in stac-fastapi-pgstac.")
kube("get", "secret", "managed-postgres", "-o", "name")
print("The managed PostgreSQL Secret is available in vc0.")

Ask the workspace operator for the ws-alice/vc0/demo kubeconfig.


{'context': 'ws-alice-vcluster-vc0-demo-87774ca8fd', 'namespace': 'stac-fastapi-pgstac'}


The managed PostgreSQL Secret is available in vc0.


## 2. Install PgSTAC and start the API

Keep the generated API password in a separate Kubernetes Secret. Reuse
that Secret on another run. Neither the notebook nor its output contains
the managed PostgreSQL password.

The migration Job uses the admin login once. The API receives only the
database host, port, database name, and its own `stac_catalog` login.
The `pgstac_ingest` role permits the registration calls in step 4.


In [2]:
if kube("get", "secret", "catalog-postgres", check=False).returncode:
    application_secret = {
        "apiVersion": "v1", "kind": "Secret",
        "metadata": {"name": "catalog-postgres", "namespace": NAMESPACE},
        "type": "Opaque",
        "stringData": {"PGUSER": "stac_catalog", "PGPASSWORD": secrets.token_urlsafe(32)},
    }
    try:
        kube("create", "-f", "-", document=application_secret)
    finally:
        application_secret["stringData"].clear()
        del application_secret

apply({
    "apiVersion": "v1", "kind": "ConfigMap",
    "metadata": {"name": "catalog-migrate", "namespace": NAMESPACE},
    "data": {"migrate.py": (MANIFEST_DIR / "migrate.py").read_text()},
})
apply({
    "apiVersion": "v1", "kind": "ConfigMap",
    "metadata": {"name": "catalog-settings", "namespace": NAMESPACE},
    "data": {"ROOT_PATH": ROOT_PATH, "LOCAL_API_ORIGIN": f"http://127.0.0.1:{LOCAL_PORT}"},
})
apply({
    "apiVersion": "v1", "kind": "ConfigMap",
    "metadata": {"name": "catalog-app", "namespace": NAMESPACE},
    "data": {"app.py": (MANIFEST_DIR / "app.py").read_text()},
})
# A finished Job has an immutable Pod template. Recreate it for a rerun.
kube("delete", "job", "catalog-migrate", "--ignore-not-found", "--wait=true")
kube("apply", "-f", str(MANIFEST_DIR / "01-migrate.yaml"))
deadline = time.monotonic() + 360
while time.monotonic() < deadline:
    status = json.loads(kube("get", "job", "catalog-migrate", "-o", "json").stdout).get("status", {})
    if status.get("failed"):
        raise RuntimeError("The PgSTAC migration Job failed. Check its Pod status and logs.")
    if status.get("succeeded"):
        break
    time.sleep(3)
else:
    raise TimeoutError("The PgSTAC migration Job did not finish.")
print("PgSTAC schema and API login are ready.")

kube("apply", "-f", str(MANIFEST_DIR / "02-api.yaml"))
kube("rollout", "restart", "deployment/catalog")
kube("rollout", "status", "deployment/catalog", "--timeout=300s")
print("STAC FastAPI is ready in vc0.")


PgSTAC schema and API login are ready.


STAC FastAPI is ready in vc0.


## 3. Open the API through Kubernetes

Start `kubectl proxy` on `127.0.0.1`. Requests use Alice's vc0 profile and the Kubernetes Service proxy. This route does not need port-forward streaming.

Keep this process running while you use the notebook. The links below open the API through the same local proxy.

In [3]:
previous_forward = globals().get("catalog_forward")
if previous_forward is not None and previous_forward.poll() is None:
    previous_forward.terminate()
    previous_forward.wait(timeout=10)
catalog_forward = subprocess.Popen(
    [*KUBECTL, "proxy", f"--port={LOCAL_PORT}", "--address=127.0.0.1", "--append-server-path"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
API_URL = f"http://127.0.0.1:{LOCAL_PORT}{PROXY_PATH}"

http = requests.Session()
http.trust_env = False
http.max_redirects = 0
for attempt in range(60):
    if catalog_forward.poll() is not None:
        raise RuntimeError("The Kubernetes proxy stopped. Check the local port and vc0 access.")
    try:
        response = http.get(API_URL + "/", timeout=3)
        if response.ok and response.headers.get("content-type", "").startswith("application/json"):
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError("The API did not become ready through the Kubernetes proxy.")

def show_links():
    base = API_URL
    display(HTML("<p>" + " · ".join(
        f'<a href="{escape(base + path, quote=True)}" target="_blank" rel="noopener">{title}</a>'
        for path, title in [("/", "API root"), ("/api.html", "API documentation"),
                            ("/collections", "Collections"), ("/collections/naip/items", "NAIP Items")]
    ) + "</p>"))

print({"api": API_URL, "namespace": NAMESPACE})
show_links()

{'api': 'http://127.0.0.1:8002/api/v1/namespaces/stac-fastapi-pgstac/services/catalog:8080/proxy', 'namespace': 'stac-fastapi-pgstac'}


## 4. Register the example catalog

Convert the three GeoParquet rows to STAC Items. Keep their source asset
links. This step copies metadata into PostgreSQL; it does not download or
upload the image files. Some source assets need a signed URL from their
original service.

Read the source Collection for its license and providers, then set its
extent to these three sample Items. Use POST for a new record and PUT for
an existing record, so another run keeps the same Item IDs.


In [4]:
sample_path = REPO_ROOT / "docs/catalog.v1.parquet"
items = [pystac.Item.from_dict({"type": "Feature", **value}, migrate=False) for value in
         stac_table_to_items(pq.read_table(sample_path))]
assert len(items) == 3 and {item.collection_id for item in items} == {"naip"}
source_url = next(link.href for link in items[0].links if link.rel == "collection")
source_response = http.get(source_url, timeout=60)
source_response.raise_for_status()
collection = pystac.Collection.from_dict(source_response.json())
collection.title = "NAIP sample catalog"
collection.description = "Three NAIP Items from the Workspace example GeoParquet catalog."
collection.extent = pystac.Extent.from_items(items)
collection.links = [link for link in collection.links
                    if link.rel not in {"self", "root", "parent", "child", "item", "items"}]
collection.add_link(pystac.Link("derived_from", source_url, media_type="application/json"))

def register(path, value):
    record_url = API_URL + path + "/" + quote(value["id"], safe="")
    existing = http.get(record_url, timeout=30)
    if existing.status_code == 404:
        response = http.post(API_URL + path, json=value, timeout=60)
    else:
        existing.raise_for_status()
        response = http.put(record_url, json=value, timeout=60)
    response.raise_for_status()
    if response.status_code not in (200, 201, 204):
        raise RuntimeError("STAC registration did not succeed.")

register("/collections", collection.to_dict())
for item in items:
    item.links = [link for link in item.links
                  if link.rel not in {"self", "root", "parent", "collection"}]
    register("/collections/naip/items", item.to_dict())
print({"source": sample_path.name, "collection": collection.id,
       "registered_items": [item.id for item in items]})
show_links()


{'source': 'catalog.v1.parquet', 'collection': 'naip', 'registered_items': ['ne_m_4110264_sw_13_060_20220827', 'ne_m_4110263_sw_13_060_20220820', 'ne_m_4110263_se_13_060_20220820']}


## 5. List Collections and Items with pystac-client

[`pystac-client`](https://pystac-client.readthedocs.io/en/stable/usage.html)
is the Python HTTP client for this STAC API. PgSTAC runs its searches in
PostgreSQL. The search calls in step 6 use a small page size to exercise pagination.


In [5]:
from pystac_client.stac_api_io import StacApiIO

stac_io = StacApiIO()
stac_io.session.trust_env = False
catalog = Client.open(API_URL, stac_io=stac_io)
print("Collections:", [value.id for value in catalog.get_collections()])
naip = catalog.get_collection("naip")
listed = list(naip.get_items())
assert {item.id for item in listed} == {item.id for item in items}
display([{"id": item.id, "datetime": item.datetime.isoformat(), "bbox": item.bbox}
         for item in listed])


Collections: ['naip']


[{'id': 'ne_m_4110264_sw_13_060_20220827',
  'datetime': '2022-08-27T16:00:00+00:00',
  'bbox': [-102.131414, 40.995619, -102.055505, 41.067376]},
 {'id': 'ne_m_4110263_sw_13_060_20220820',
  'datetime': '2022-08-20T16:00:00+00:00',
  'bbox': [-102.256294, 40.995689, -102.180634, 41.067303]},
 {'id': 'ne_m_4110263_se_13_060_20220820',
  'datetime': '2022-08-20T16:00:00+00:00',
  'bbox': [-102.193354, 40.995681, -102.118573, 41.067312]}]

## 6. Search by area, date, and Item ID

The small bounding box intersects only the eastern sample Item. The date
filter selects the two Items from 20 August. Check an empty result as well
as matches, so a filter that the server ignores causes a failure.


In [6]:
all_matches = list(catalog.search(collections=["naip"], limit=2).items())
assert {item.id for item in all_matches} == {item.id for item in items}

spatial = list(catalog.search(
    collections=["naip"], bbox=[-102.08, 41.01, -102.06, 41.05],
).items())
assert [item.id for item in spatial] == ["ne_m_4110264_sw_13_060_20220827"]

dated = list(catalog.search(
    collections=["naip"], datetime="2022-08-20T00:00:00Z/2022-08-20T23:59:59Z",
).items())
assert {item.id for item in dated} == {
    "ne_m_4110263_sw_13_060_20220820", "ne_m_4110263_se_13_060_20220820",
}

selected = list(catalog.search(collections=["naip"], ids=[items[0].id], method="GET").items())
assert [item.id for item in selected] == [items[0].id]
empty = list(catalog.search(collections=["naip"], bbox=[16, 48, 17, 49]).items())
assert not empty

display({"all": [item.id for item in all_matches],
         "area": [item.id for item in spatial],
         "date": [item.id for item in dated],
         "id_via_get": [item.id for item in selected], "outside_area": len(empty)})


{'all': ['ne_m_4110264_sw_13_060_20220827',
  'ne_m_4110263_sw_13_060_20220820',
  'ne_m_4110263_se_13_060_20220820'],
 'area': ['ne_m_4110264_sw_13_060_20220827'],
 'date': ['ne_m_4110263_sw_13_060_20220820',
  'ne_m_4110263_se_13_060_20220820'],
 'id_via_get': ['ne_m_4110264_sw_13_060_20220827'],
 'outside_area': 0}

## 7. Inspect an Item

Fetch an Item by its Collection and ID. Compare its geometry and asset
metadata with the sample to check that registration kept their values.
The API omits null optional fields in band descriptions. Ignore those null
object fields in the comparison. The API supplies new navigation links
for this local catalog.


In [7]:
def without_null_fields(value):
    if isinstance(value, dict):
        return {key: without_null_fields(part) for key, part in value.items() if part is not None}
    if isinstance(value, list):
        return [without_null_fields(part) for part in value]
    return value


stored = naip.get_item(items[0].id)
assert stored is not None
assert stored.geometry == items[0].geometry
assert without_null_fields(stored.to_dict()["assets"]) == without_null_fields(items[0].to_dict()["assets"])
display({"id": stored.id, "collection": stored.collection_id,
         "assets": {name: asset.href for name, asset in stored.assets.items()}})
print("All three Items and the listing, search, and metadata checks passed.")


{'id': 'ne_m_4110264_sw_13_060_20220827',
 'collection': 'naip',
 'assets': {'image': './v002/ne/2022/ne_060cm_2022/41102/m_4110264_sw_13_060_20220827.tif',
  'tilejson': 'https://planetarycomputer.microsoft.com/api/data/v1/item/tilejson.json?collection=naip&item=ne_m_4110264_sw_13_060_20220827&assets=image&asset_bidx=image%7C1%2C2%2C3&format=png',
  'thumbnail': './v002/ne/2022/ne_060cm_2022/41102/m_4110264_sw_13_060_20220827.200.jpg',
  'rendered_preview': 'https://planetarycomputer.microsoft.com/api/data/v1/item/preview.png?collection=naip&item=ne_m_4110264_sw_13_060_20220827&assets=image&asset_bidx=image%7C1%2C2%2C3&format=png'}}

All three Items and the listing, search, and metadata checks passed.


## 8. Read PostgreSQL directly

The [psql client manifest](stac-fastapi-pgstac/03-psql.yaml) uses the same managed Secret. The code below runs a short query in vc0 and reads its Pod log. The password stays in Kubernetes.

In [8]:
psql_pod = {
    "apiVersion": "v1", "kind": "Pod",
    "metadata": {"name": "xyz", "namespace": NAMESPACE},
    "spec": {
        "restartPolicy": "Never",
        "containers": [{
            "name": "psql", "image": "postgres:17",
            "command": ["psql", "-v", "ON_ERROR_STOP=1", "-c",
                        "SELECT current_database(); SELECT id FROM pgstac.collections ORDER BY id; "
                        "SELECT collection, id FROM pgstac.items ORDER BY collection, id;"],
            "env": [
                {"name": variable, "valueFrom": {"secretKeyRef": {
                    "name": "managed-postgres", "key": key,
                }}} for variable, key in [
                    ("PGHOST", "host"), ("PGPORT", "port"),
                    ("PGUSER", "user"), ("PGPASSWORD", "password"),
                ]
            ] + [{"name": "PGDATABASE", "value": "catalog"},
                 {"name": "PGSSLMODE", "value": "require"}],
        }],
    },
}
kube("delete", "pod", "xyz", "--ignore-not-found", "--wait=true")
kube("create", "-f", "-", document=psql_pod)
try:
    for attempt in range(120):
        phase = kube("get", "pod", "xyz", "-o", "jsonpath={.status.phase}").stdout
        if phase == "Succeeded":
            break
        if phase == "Failed":
            raise RuntimeError("The psql query failed. Check the client Pod.")
        time.sleep(1)
    else:
        raise TimeoutError("The psql query did not finish.")
    query_output = kube("logs", "xyz").stdout
    assert "catalog" in query_output and "naip" in query_output
    assert all(item.id in query_output for item in items)
    print(query_output.strip())
finally:
    kube("delete", "pod", "xyz", "--ignore-not-found")


current_database 
------------------
 catalog
(1 row)

  id  
------
 naip
(1 row)

 collection |               id                
------------+---------------------------------
 naip       | ne_m_4110263_se_13_060_20220820
 naip       | ne_m_4110263_sw_13_060_20220820
 naip       | ne_m_4110264_sw_13_060_20220827
(3 rows)


## 9. Stop the local proxy

Set `STOP_FORWARD = True` when you finish. The API and database stay available.

In [9]:
STOP_FORWARD = False
if STOP_FORWARD and catalog_forward.poll() is None:
    catalog_forward.terminate()
    catalog_forward.wait(timeout=10)
    print("The Kubernetes proxy stopped.")